# SAM3 다중 텍스트 프롬프트 — 영상 추론

여러 개념을 동시에 텍스트 프롬프트로 넘겨 SAM3가 세그먼트한 결과를 영상으로 저장한다.

- 초당 5프레임 추출
- 프롬프트별 다른 색상으로 마스크 오버레이
- 결과를 mp4 영상으로 저장

In [1]:
import cv2
import numpy as np
from pathlib import Path
from datetime import datetime
from ultralytics.models.sam import SAM3SemanticPredictor

In [2]:
VIDEO_PATH = "videos/test.mp4"

_ts          = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_VIDEO = f"sam3_multi_output_{_ts}.mp4"

# 다중 텍스트 프롬프트 — 순서대로 색상이 할당됨
TEXT_PROMPTS = [
    "elevator stair",
    "yellow safety guard",
    "handrail",
    "person",
]

# 프롬프트별 BGR 색상
COLORS = [
    (255, 100,   0),   # elevator stair  → 파란 계열
    (  0, 220, 255),   # yellow guard    → 노란 계열
    (  0, 200,   0),   # handrail        → 초록
    (100,   0, 255),   # person          → 보라
]

TARGET_FPS = 5        # 추출 프레임 수 (초당)
CONF       = 0.25

print(f"출력 파일명: {OUTPUT_VIDEO}")

출력 파일명: sam3_multi_output_20260513_015424.mp4


## Step 1 — SAM3SemanticPredictor 초기화

In [3]:
overrides = dict(
    conf=CONF,
    task="segment",
    mode="predict",
    model="sam3.pt",
    half=True,
    save=False,
)
predictor = SAM3SemanticPredictor(overrides=overrides)
print("SAM3SemanticPredictor 초기화 완료")
print(f"프롬프트: {TEXT_PROMPTS}")

SAM3SemanticPredictor 초기화 완료
프롬프트: ['elevator stair', 'yellow safety guard', 'handrail', 'person']


## Step 2 — 프레임 추출 (초당 5장)

In [4]:
cap = cv2.VideoCapture(VIDEO_PATH)
src_fps = 60.0
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fh      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fw      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

# 몇 프레임마다 1장 추출할지 계산
step = max(1, round(src_fps / TARGET_FPS))

frames = []
idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if idx % step == 0:
        frames.append((idx, frame))
    idx += 1
cap.release()

print(f"원본: {total}프레임  {src_fps:.1f}fps  ({fw}×{fh})")
print(f"추출: {len(frames)}프레임  (step={step}, 실효 {src_fps/step:.1f}fps)")

원본: 649프레임  60.0fps  (1080×1920)
추출: 55프레임  (step=12, 실효 5.0fps)


## Step 3 — 다중 텍스트 프롬프트 추론 & 결과 영상 저장

In [5]:
writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    TARGET_FPS,
    (fw, fh),
)

for proc_idx, (orig_idx, frame) in enumerate(frames):
    predictor.set_image(frame)
    results = predictor(text=TEXT_PROMPTS)

    out = frame.copy()
    counts = {p: 0 for p in TEXT_PROMPTS}

    if results and results[0].masks is not None:
        r = results[0]
        masks   = r.masks.data.cpu().numpy().astype(np.uint8)  # bool → uint8
        cls_ids = r.boxes.cls.cpu().numpy().astype(int) if r.boxes is not None else []

        for mask, cls_id in zip(masks, cls_ids):
            mask_r = cv2.resize(mask, (fw, fh), interpolation=cv2.INTER_NEAREST)
            color  = COLORS[cls_id % len(COLORS)]

            overlay = out.copy()
            overlay[mask_r > 0] = color
            cv2.addWeighted(overlay, 0.40, out, 0.60, 0, out)

            prompt_name = TEXT_PROMPTS[cls_id] if cls_id < len(TEXT_PROMPTS) else str(cls_id)
            counts[prompt_name] = counts.get(prompt_name, 0) + 1

    # HUD — 프롬프트별 감지 수
    hud = out.copy()
    hud_h = 30 + 34 * len(TEXT_PROMPTS)
    cv2.rectangle(hud, (0, 0), (fw, hud_h), (0, 0, 0), -1)
    cv2.addWeighted(hud, 0.50, out, 0.50, 0, out)

    cv2.putText(out, f"frame {orig_idx}  ({proc_idx+1}/{len(frames)})",
                (12, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (220, 220, 220), 1, cv2.LINE_AA)
    for i, prompt in enumerate(TEXT_PROMPTS):
        color = COLORS[i % len(COLORS)]
        label = f"  {prompt}: {counts.get(prompt, 0)}"
        cv2.putText(out, label, (12, 56 + i * 34),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)

    writer.write(out)

    if proc_idx % 10 == 0:
        print(f"  [{proc_idx+1:3d}/{len(frames)}] frame={orig_idx}  {dict(counts)}")

writer.release()
print(f"\n완료: {OUTPUT_VIDEO}")

Ultralytics 8.4.48  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

0: 644x644 3 elevator stairs, 4 handrails, 237.7ms
Speed: 8.9ms preprocess, 237.7ms inference, 16.0ms postprocess per image at shape (1, 3, 644, 644)
  [  1/55] frame=0  {'elevator stair': 3, 'yellow safety guard': 0, 'handrail': 4, 'person': 0}
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

0: 644x644 1 elevator stair, 5 handrails, 126.3ms
Speed: 2.2ms preprocess, 126.3ms inference, 1.8ms postprocess per image at shape (1, 3, 644, 644)
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

0: 644x644 6 elevator stairs, 5 handrails, 127.9ms
Speed: 2.1ms preprocess, 127.9ms inference, 1.7ms postprocess per image at shape (1, 3, 644, 644)
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

0: 644x644 1 elevator stair, 5 handrails, 127.4ms
Speed: 2.3ms p